# Phase 3B: multilingual fine-tuning

XLM-R Base fine-tuned on all five languages; the epoch with the best validation macro-F1 is kept.

All numbers come from the exported run artifacts in `results/`. The current cohort is the provisional `gemini-v2` `span_checked` release (322 test question groups, human review pending), so results must be labelled provisional.

In [ ]:
import csv, json
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS = Path.cwd().parent / "results" if Path.cwd().name == "notebooks" else Path("phase_03_multilingual_finetuning/results")
LANGS = ["en", "hi", "mr", "bn", "ta"]

def load_json(name):
    return json.loads((RESULTS / name).read_text(encoding="utf-8"))

def load_csv(name):
    with (RESULTS / name).open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))

def show(rows, fields=None, digits=2):
    """Print a plain-text table without extra dependencies."""
    fields = fields or list(rows[0])
    def fmt(value):
        try:
            return f"{float(value):.{digits}f}" if "." in str(value) else str(value)
        except ValueError:
            return str(value)
    cells = [[fmt(row.get(f, "")) for f in fields] for row in rows]
    widths = [max(len(f), *(len(c[i]) for c in cells)) for i, f in enumerate(fields)]
    print("  ".join(f.ljust(w) for f, w in zip(fields, widths)))
    for c in cells:
        print("  ".join(v.ljust(w) for v, w in zip(c, widths)))

def require(*names):
    missing = [n for n in names if not (RESULTS / n).exists()]
    if missing:
        raise FileNotFoundError("Run scripts/export_results.py first. Missing: " + ", ".join(missing))

In [ ]:
require("phase_3b/training_logs.csv", "phase_3b/validation_metrics.csv", "phase_3b/metrics.json")
logs = load_csv("phase_3b/training_logs.csv")
train = [r for r in logs if r["train_loss"]]
val = [r for r in logs if r["val_loss"]]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([float(r["epoch"]) for r in train], [float(r["train_loss"]) for r in train], label="Training loss")
ax.plot([float(r["epoch"]) for r in val], [float(r["val_loss"]) for r in val], marker="o", label="Validation loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Span loss"); ax.legend(); ax.set_title("Phase 3B loss")
plt.show()

In [ ]:
validation = load_csv("phase_3b/validation_metrics.csv")
fig, ax = plt.subplots(figsize=(7, 4))
for lang in LANGS:
    rows = [r for r in validation if r["language"] == lang]
    ax.plot([float(r["epoch"]) for r in rows], [float(r["f1"]) for r in rows], marker="o", label=lang.upper())
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation F1 (%)"); ax.legend(); ax.set_title("Validation F1 convergence")
plt.show()
show([r for r in validation if r["epoch"] == validation[-1]["epoch"]])

## Test results and compute

In [ ]:
show(load_csv("phase_3b/language_wise_results.csv"))
compute = load_json("phase_3b/computational_statistics.json")
training = compute["training"]
for key in ("device_name", "parameters", "training_rows", "tokenized_training_features", "optimizer_steps",
            "training_seconds", "peak_gpu_memory_bytes", "validation_best_macro_f1", "selected_epoch_checkpoint"):
    print(f"{key}: {training.get(key)}")
print("average_inference_ms_per_question_row:", compute["average_inference_ms_per_question_row"])